# Exact DMD

`dmd.exactDMD` with amplitudes fitted to the whole record by `dmd.DMD_amplitudes_optimal`. Periods near or beyond the record length are indicative only.

Data: GLORYS04 SSH (364 days) and surface U and V in the Gulf Stream (731 days, 30-45N, 75-45W). Animations show the anomaly beside its reconstruction.

In [ ]:
import numpy as np
from showcase import *
from modredkit import dmd, pod

RANK = 50

## SSH

In [ ]:
A, area, wet = load_ssh()
print("SSH anomalies", A.shape)
mds, eig, _ = dmd.exactDMD(A[:, :-1], A[:, 1:], RANK)
ct = dmd.DMD_continuous_eig(eig, DT)
b = dmd.DMD_amplitudes_optimal(mds, eig, A)
print(f"{np.sum(np.abs(eig) > 1)} of {eig.size} modes grow; relative error {relative_error(rebuild(mds, b, eig, A.shape[1]), A):.3f}")
show_spectrum(ct, b, "SSH: exact DMD spectrum")
top = np.argsort(-np.abs(b)*np.linalg.norm(mds, axis=0))[:2]
for k in top:
    show_maps([(mds[:, k]*b[k]).real], [f"SSH mode {k}, period {2*np.pi/max(abs(ct[k].imag), 1e-12):.0f} d"], [wet])

In [ ]:
rec = rebuild(mds, b, eig, A.shape[1])
animate([A, rec], ["SSH anomaly", "Exact DMD, 50 modes"], wet, units="m")

## Gulf Stream U and V

U and V are stacked into one state vector, so they share eigenvalues.

In [ ]:
U, V, wet_u, wet_v = load_gulfstream()
Z = np.vstack((U, V))
nu = U.shape[0]
print("U", U.shape, "V", V.shape)
mds, eig, _ = dmd.exactDMD(Z[:, :-1], Z[:, 1:], RANK)
ct = dmd.DMD_continuous_eig(eig, DT)
b = dmd.DMD_amplitudes_optimal(mds, eig, Z)
print(f"{np.sum(np.abs(eig) > 1)} of {eig.size} modes grow; relative error {relative_error(rebuild(mds, b, eig, Z.shape[1]), Z):.3f}")
show_spectrum(ct, b, "Gulf Stream: exact DMD spectrum")
top = np.argsort(-np.abs(b)*np.linalg.norm(mds, axis=0))[:2]
for k in top:
    f = (mds[:, k]*b[k]).real
    show_maps([f[:nu], f[nu:]], [f"U, mode {k}, period {2*np.pi/max(abs(ct[k].imag), 1e-12):.0f} d", "V"], [wet_u, wet_v])

In [ ]:
rec = rebuild(mds, b, eig, Z.shape[1])
animate([U, rec[:nu]], ["U anomaly", "Exact DMD, 50 modes"], wet_u, step=12, units="m/s")